# ML-02 - Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Rafay-246/internship-flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order**. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing **Lane 2: Refresh / Content Opportunity Scoring**. This lane fits the starter data because the dataset already has page-level search, engagement, age, freshness, and trend signals. The practical goal is not to automatically edit pages. The goal is to build a ranked review queue that helps a content reviewer decide which pages deserve attention first.

In [1]:
import pandas as pd

csv_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

number_of_rows = len(df)
number_of_clients = df["client_id"].nunique()

print("Loaded rows:", number_of_rows)
print("Pseudonymized clients:", number_of_clients)


Loaded rows: 30000
Pseudonymized clients: 32


## 2. The question: decision, action, cost of a wrong call

My research question is: **Which content pages should a reviewer check first for refresh, CTR review, engagement review, expansion, or monitoring?**

The decision this improves is the order of page review when the team has limited time. The person acting on the output is a content reviewer, editor, or SEO analyst. The action is to inspect the highest-ranked pages first and decide whether to refresh, improve metadata, improve on-page content, expand the page, or monitor it.

A wrong recommendation has two main costs. A false positive wastes reviewer time on a page that did not need action. A false negative misses a page that may be losing search opportunity. Because of that, the output should be treated as decision support, not an automatic publishing instruction.

In [2]:
declining_pages = df[df["trend_direction"] == "down"]
declining_with_demand = declining_pages[declining_pages["impressions_90d"] >= 100]

visible_pages = df[df["impressions_90d"] >= 500]

visible_pages_with_position = visible_pages[visible_pages["avg_position"] > 0]
visible_pages_position_1_to_20 = visible_pages_with_position[
    visible_pages_with_position["avg_position"] <= 20
]
low_ctr_visible_pages = visible_pages_position_1_to_20[
    visible_pages_position_1_to_20["ctr"] < 0.5
]

print("Declining pages with at least 100 impressions:", len(declining_with_demand))
print("Visible pages with at least 500 impressions:", len(visible_pages))
print("Visible pages in position 1-20 with CTR under 0.5%:", len(low_ctr_visible_pages))


Declining pages with at least 100 impressions: 13152
Visible pages with at least 500 impressions: 16726
Visible pages in position 1-20 with CTR under 0.5%: 9759


## 3. Quick look at the data (2-3 real numbers)

The starter data gives enough evidence to make Lane 2 worth exploring. It has **30,000** page-level rows across **32** pseudonymized clients. It also has **16,262** pages marked as `down` by the starter trend label, which is about **54.2%** of the starter dataset. Among those, **13,152** declining pages still have at least 100 impressions, so many pages have enough search exposure to make review meaningful.

The data also supports action-specific review. There are **9,759** visible pages in average position 1-20 with at least 500 impressions and CTR below 0.5%. That means a refresh queue can include reason codes like declining with demand, low CTR, and engagement review instead of giving a black-box score only.

In [3]:
declining_count = len(declining_pages)
declining_rate = declining_count / number_of_rows

print("Content rows:", number_of_rows)
print("Pseudonymized clients:", number_of_clients)
print("Declining rows:", declining_count)
print("Declining rate:", round(declining_rate, 3))
print("Declining rows with at least 100 impressions:", len(declining_with_demand))
print("Low-CTR visible rows:", len(low_ctr_visible_pages))


Content rows: 30000
Pseudonymized clients: 32
Declining rows: 16262
Declining rate: 0.542
Declining rows with at least 100 impressions: 13152
Low-CTR visible rows: 9759


## 4. Careful words: what I can and can't claim

This project can claim that the data shows **observed signals** that help rank pages for human review. It can say that some pages look like stronger review candidates because they combine signs such as search demand, decline, low CTR, weak engagement, age, or freshness risk.

This project cannot claim that refreshing a page will cause recovery. It also cannot claim to predict Google, prove Google ranking factors, reveal private client details, or prove that a page is bad. The safest claim is: **this is a decision-support ranking that helps reviewers choose which pages to inspect first.**

In [4]:
private_like_columns = []

for column_name in df.columns:
    lower_name = column_name.lower()

    if "url" in lower_name:
        private_like_columns.append(column_name)
    elif "domain" in lower_name:
        private_like_columns.append(column_name)
    elif "title" in lower_name:
        private_like_columns.append(column_name)
    elif "query" in lower_name:
        private_like_columns.append(column_name)
    elif "client_name" in lower_name:
        private_like_columns.append(column_name)


excluded_from_features = ["content_id", "client_id", "trend_direction", "trend_pct"]

print("Private/raw identifying columns found:", private_like_columns)
print("Fields to exclude from model features:", excluded_from_features)


Private/raw identifying columns found: []
Fields to exclude from model features: ['content_id', 'client_id', 'trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.